# 🔍 Project 8.2 — Open Addressing Probe Table
**DSA for Mechatronics · Week 08 — Hash Tables & Dictionaries**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
from collections import Counter, defaultdict, OrderedDict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A PLC stores calibration constants for actuator channels in a **fixed-size hash table**.
Because memory is tight, all entries are stored **in the table array itself** (open addressing).

You will implement two probing strategies and compare them:
- **Linear probing**: check slot i, i+1, i+2, … — simple but causes **primary clustering**
- **Quadratic probing**: check i, i+1², i+2², … — spreads entries, reduces clustering

A deleted slot must store a **TOMBSTONE** marker so that probe chains remain intact.


## Step 1 — Implement linear probing hash table

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
TABLE_SIZE  = random.choice([17, 19, 23, 29, 31])   # prime capacity
N_INSERT    = random.randint(10, int(TABLE_SIZE * 0.7))
N_DELETE_OA = random.randint(2, min(4, N_INSERT // 3))
N_LOOKUP_OA = random.randint(5, 8)

CHANNEL_NAMES = [f"CH{i:03d}" for i in range(1, 60)]
CALIB_VALUES  = [round(random.uniform(0.01, 9.99), 3) for _ in range(100)]

keys   = random.sample(CHANNEL_NAMES, N_INSERT)
values = [random.choice(CALIB_VALUES) for _ in range(N_INSERT)]

DELETED = object()   # sentinel tombstone

class LinearProbingMap:
    """
    Open addressing hash table with linear probing.
    slots[i] = None     → empty (never used)
    slots[i] = DELETED  → tombstone (was deleted, skip but continue probing)
    slots[i] = (k, v)   → occupied
    """
    def __init__(self, capacity):
        self.capacity  = capacity
        self.slots     = [None] * capacity
        self.size      = 0
        self.probes_put  = 0   # total probe steps during all puts
        self.probes_get  = 0   # total probe steps during all gets

    def _hash(self, key):
        h = 0
        for ch in str(key):
            h = (h * 31 + ord(ch)) & 0xFFFFFFFF
        return h % self.capacity

    @property
    def load_factor(self):
        return self.size / self.capacity

    def put(self, key, value):
        """Insert/update with linear probing. Raise if table is full."""
        start = self._hash(key)
        first_tomb = None
        for step in range(self.capacity):
            i = (start + step) % self.capacity
            self.probes_put += 1
            slot = self.slots[i]
            if slot is None:
                # empty slot — insert here (or at first tombstone)
                target = first_tomb if first_tomb is not None else i
                self.slots[target] = (key, value)
                self.size += 1
                return
            if slot is DELETED:
                if first_tomb is None:
                    first_tomb = i
            elif slot[0] == key:
                self.slots[i] = (key, value)   # update
                return
        if first_tomb is not None:
            self.slots[first_tomb] = (key, value)
            self.size += 1
        else:
            raise OverflowError("Hash table is full")

    def get(self, key, default=None):
        """Lookup with linear probing. Returns default if not found."""
        start = self._hash(key)
        for step in range(self.capacity):
            i = (start + step) % self.capacity
            self.probes_get += 1
            slot = self.slots[i]
            if slot is None:
                return default   # definitively absent
            if slot is not DELETED and slot[0] == key:
                return slot[1]
        return default

    def delete(self, key):
        """Replace entry with TOMBSTONE. O(1) avg."""
        start = self._hash(key)
        for step in range(self.capacity):
            i = (start + step) % self.capacity
            slot = self.slots[i]
            if slot is None:
                return False
            if slot is not DELETED and slot[0] == key:
                self.slots[i] = DELETED
                self.size -= 1
                return True
        return False

    def tombstone_count(self):
        return sum(1 for s in self.slots if s is DELETED)

lp = LinearProbingMap(TABLE_SIZE)

print(f"Linear probing table parameters:")
print(f"  Table capacity    : {TABLE_SIZE}  (prime)")
print(f"  Entries to insert : {N_INSERT}")
print(f"  Target load factor: {N_INSERT/TABLE_SIZE:.2f}")
print()

for k, v in zip(keys, values):
    lp.put(k, v)

print(f"After inserting {N_INSERT} entries:")
print(f"  {'Slot':>5}  {'Key':<8}  {'Value':>8}  Status")
print(f"  {'─'*5}  {'─'*8}  {'─'*8}  {'─'*12}")
for i, slot in enumerate(lp.slots):
    if slot is None:
        print(f"  {i:5d}  {'—':<8}  {'':>8}  empty")
    elif slot is DELETED:
        print(f"  {i:5d}  {'TOMBSTONE':<8}  {'':>8}  deleted")
    else:
        k, v = slot
        print(f"  {i:5d}  {k:<8}  {v:8.3f}  occupied")


## Step 2 — Probe count and deletion analysis

In [ ]:
# Lookup queries (some present, some absent)
present_q = random.sample(keys, N_LOOKUP_OA // 2)
absent_q  = [f"CH{random.randint(100,200)}" for _ in range(N_LOOKUP_OA // 2)]
all_q     = present_q + absent_q
random.shuffle(all_q)

print(f"Lookup results ({len(all_q)} queries):")
print(f"  {'Key':<8}  {'Found':>6}  {'Value':>8}")
print(f"  {'─'*8}  {'─'*6}  {'─'*8}")
for q in all_q:
    val = lp.get(q)
    if val is not None:
        print(f"  {q:<8}  {'YES':>6}  {val:8.3f}")
    else:
        print(f"  {q:<8}  {'NO':>6}  {'—':>8}")

# Delete some entries
to_del = random.sample(keys, N_DELETE_OA)
print(f"\nDeleting {N_DELETE_OA} entries: {to_del}")
for k in to_del:
    lp.delete(k)

print(f"\nAfter deletion:")
print(f"  Size              : {lp.size}")
print(f"  Load factor       : {lp.load_factor:.3f}")
print(f"  Tombstones        : {lp.tombstone_count()}")
print(f"  Put probes total  : {lp.probes_put}")
print(f"  Get probes total  : {lp.probes_get}")
avg_put = lp.probes_put / N_INSERT
avg_get = lp.probes_get / len(all_q)
print(f"  Avg probes/put    : {avg_put:.2f}")
print(f"  Avg probes/get    : {avg_get:.2f}")


## Step 3 — Quadratic probing comparison

In [ ]:
class QuadraticProbingMap:
    """
    Open addressing with quadratic probing.
    step i uses offset i² : slots checked = h, h+1, h+4, h+9, h+16, …
    Guarantees full coverage when capacity is prime and load < 0.5.
    """
    def __init__(self, capacity):
        self.capacity = capacity
        self.slots    = [None] * capacity
        self.size     = 0
        self.probes_put = 0
        self.probes_get = 0

    def _hash(self, key):
        h = 0
        for ch in str(key):
            h = (h * 31 + ord(ch)) & 0xFFFFFFFF
        return h % self.capacity

    @property
    def load_factor(self):
        return self.size / self.capacity

    def put(self, key, value):
        start = self._hash(key)
        first_tomb = None
        for step in range(self.capacity):
            i = (start + step * step) % self.capacity
            self.probes_put += 1
            slot = self.slots[i]
            if slot is None:
                target = first_tomb if first_tomb is not None else i
                self.slots[target] = (key, value)
                self.size += 1
                return
            if slot is DELETED:
                if first_tomb is None: first_tomb = i
            elif slot[0] == key:
                self.slots[i] = (key, value); return
        if first_tomb is not None:
            self.slots[first_tomb] = (key, value); self.size += 1
        else:
            raise OverflowError("Hash table is full")

    def get(self, key, default=None):
        start = self._hash(key)
        for step in range(self.capacity):
            i = (start + step * step) % self.capacity
            self.probes_get += 1
            slot = self.slots[i]
            if slot is None: return default
            if slot is not DELETED and slot[0] == key: return slot[1]
        return default

qp = QuadraticProbingMap(TABLE_SIZE)
for k, v in zip(keys, values):
    qp.put(k, v)
# same lookups
for q in all_q:
    qp.get(q)

print(f"Probing strategy comparison (same {N_INSERT} inserts + {len(all_q)} lookups):")
print(f"  {'Metric':<28}  {'Linear':>10}  {'Quadratic':>12}")
print(f"  {'─'*28}  {'─'*10}  {'─'*12}")
print(f"  {'Total put probes':<28}  {lp.probes_put:10d}  {qp.probes_put:12d}")
print(f"  {'Total get probes':<28}  {lp.probes_get:10d}  {qp.probes_get:12d}")
print(f"  {'Avg probes/put':<28}  {lp.probes_put/N_INSERT:10.2f}  {qp.probes_put/N_INSERT:12.2f}")
print(f"  {'Avg probes/get':<28}  {lp.probes_get/len(all_q):10.2f}  {qp.probes_get/len(all_q):12.2f}")
print(f"  {'Load factor':<28}  {lp.load_factor:10.3f}  {qp.load_factor:12.3f}")

winner = "Linear" if lp.probes_put + lp.probes_get < qp.probes_put + qp.probes_get else "Quadratic"
print(f"\n  Fewer total probes: {winner}")


## Step 4 — Cluster analysis on linear probing table

In [ ]:
def find_clusters(slots):
    """
    Find all contiguous runs of non-empty slots (occupied or tombstone).
    Returns list of cluster lengths.
    """
    n        = len(slots)
    clusters = []
    i        = 0
    while i < n:
        if slots[i] is not None:   # start of a cluster
            length = 0
            while i < n and slots[i] is not None:
                length += 1
                i += 1
            clusters.append(length)
        else:
            i += 1
    return clusters

clusters = find_clusters(lp.slots)
from collections import Counter
cluster_dist = Counter(clusters)

print(f"Linear probing cluster analysis:")
print(f"  Total clusters    : {len(clusters)}")
print(f"  Avg cluster size  : {sum(clusters)/len(clusters):.2f}" if clusters else "  No clusters")
print(f"  Max cluster size  : {max(clusters) if clusters else 0}")
print()
print(f"  {'Cluster size':>13}  {'Count':>7}  Bar")
print(f"  {'─'*13}  {'─'*7}  {'─'*16}")
for sz in sorted(cluster_dist):
    bar = "█" * cluster_dist[sz]
    print(f"  {sz:13d}  {cluster_dist[sz]:7d}  {bar}")

longest_cluster = max(clusters) if clusters else 0
print(f"\n  Primary clustering note:")
if longest_cluster >= 4:
    print(f"  ⚠  Cluster of length {longest_cluster} detected — may slow lookups")
else:
    print(f"  ✅ No severe clustering (max={longest_cluster})")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " OPEN ADDRESSING PROBE TABLE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TABLE PARAMETERS" + " "*(W-18) + "║")
print(f"║  {'Capacity (prime)':<28}: {TABLE_SIZE:<26}║")
print(f"║  {'Entries inserted':<28}: {N_INSERT:<26}║")
print(f"║  {'Load factor':<28}: {lp.load_factor:.3f}{'':<22}║")
print(f"║  {'Tombstones after delete':<28}: {lp.tombstone_count():<26}║")
print("╠" + "═"*W + "╣")
print("║  PROBE COUNTS" + " "*(W-14) + "║")
print(f"║  {'Linear — avg probes/put':<28}: {lp.probes_put/N_INSERT:.2f}{'':<22}║")
print(f"║  {'Linear — avg probes/get':<28}: {lp.probes_get/len(all_q):.2f}{'':<22}║")
print(f"║  {'Quadratic — avg probes/put':<28}: {qp.probes_put/N_INSERT:.2f}{'':<22}║")
print(f"║  {'Quadratic — avg probes/get':<28}: {qp.probes_get/len(all_q):.2f}{'':<22}║")
print(f"║  {'Fewer total probes':<28}: {winner:<26}║")
print("╠" + "═"*W + "╣")
print("║  CLUSTER ANALYSIS" + " "*(W-18) + "║")
print(f"║  {'Clusters found':<28}: {len(clusters):<26}║")
print(f"║  {'Max cluster length':<28}: {longest_cluster:<26}║")
clust_avg = round(sum(clusters)/len(clusters),2) if clusters else 0
print(f"║  {'Avg cluster length':<28}: {clust_avg:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which hash table concept did you find most important, and why?**
Pick one technique (collision resolution, load factor, LRU cache, rolling hash…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
